In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Train

In [ ]:
"""
Module 3 — Music Descriptor Prediction  (Kaggle-ready, GPU)
============================================================
Reads  : /kaggle/working/narrative_vectors_train.pt  ← from Module 2
         /kaggle/working/narrative_vectors_val.pt
         /kaggle/working/narrative_vectors_test.pt
         /kaggle/working/scene_vectors.pt             ← from Module 1
         /kaggle/input/datasets/donbosoc/extended-movie-dataset

INPUT ARCHITECTURE (313-d)
--------------------------
  M1 ground truth labels (41-d):
    emotional_valence      one-hot 4
    conflict_nature        one-hot 6
    acoustic_space         one-hot 6
    reality_layer          one-hot 5
    score_dynamic_shape    one-hot 4
    scene_interaction_tone one-hot 5
    pacing_norm            scalar 1
    action_norm            scalar 1
    tension_norm           scalar 1
    arousal_score          scalar 1
    emotion_tags           7-hot  7
    ──────────────────────────────
    Subtotal               41

  M2 ground truth labels (16-d):
    tension_level          scalar 1
    arousal_level          scalar 1
    emotional_shift        scalar 1
    narrative_arc          one-hot 5
    foreshadowing_type     one-hot 2
    transition_type        one-hot 5
    ──────────────────────────────
    Subtotal               16

  context_vector from M2  (256-d):
    cross-scene temporal embedding
    ──────────────────────────────
    Subtotal               256

  Total input              313-d

8 prediction heads:
    REG  tempo_bpm
    REG  musical_valence
    CLS  tonality           3-class
    CLS  harmonic_style     7-class
    CLS  dynamic_shape_m4   8-class
    CLS  rhythm_style       6-class
    CLS  texture            5-class
    ML   orchestration      14-hot
"""

# ── Standard library ──────────────────────────────────────────────────────────
import os, re, json, glob, warnings, textwrap, random
import numpy as np
from typing import Dict, List, Optional, Tuple

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── HuggingFace ───────────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import HfApi, login, hf_hub_download

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error,
    r2_score, precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# =============================================================================
# CREDENTIALS
# =============================================================================
from kaggle_secrets import UserSecretsClient
secrets        = UserSecretsClient()
HF_READ_TOKEN  = secrets.get_secret("HF_READ_TOKEN")
HF_WRITE_TOKEN = secrets.get_secret("HF_WRITE_TOKEN")

HF_M1_REPO = "suyashnpande/scene-perception-m1-harshal"
HF_M2_REPO = "suyashnpande/narrative-context-m2-harshal-replayed"
HF_M3_REPO = "suyashnpande/music-descriptor-m3"

# =============================================================================
# CONFIG
# =============================================================================
CFG = dict(
    data_dir          = "/kaggle/input/datasets/donbosoc/extended-movie-dataset",
    scene_vectors     = "/kaggle/working/scene_vectors.pt",
    narrative_vectors_train = "/kaggle/working/narrative_vectors_train.pt",
    narrative_vectors_val   = "/kaggle/working/narrative_vectors_val.pt",
    narrative_vectors_test  = "/kaggle/working/narrative_vectors_test.pt",
    output_dir        = "/kaggle/working",
    ckpt_name         = "m3_best.pt",

    # M1 settings — for regenerating scene_vectors.pt if missing
    m1_backbone   = "distilroberta-base",
    m1_embed_dim  = 256,
    m1_dropout    = 0.2,
    m1_max_length = 512,

    # M2 settings — for regenerating narrative_vectors if missing
    m2_window_size    = 5,
    m2_scene_feat_dim = 304,
    m2_token_dim      = 256,
    m2_n_heads        = 8,
    m2_n_layers       = 4,
    m2_ffn_dim        = 512,
    m2_tf_dropout     = 0.1,
    m2_proj_dropout   = 0.2,
    m2_head_dropout   = 0.2,

    # M3 Architecture
    input_dim         = 314,   # 41 (M1 labels) + 16 (M2 labels) + 256 (context_vector)
    proj_dim          = 256,
    head_hidden       = 128,
    proj_dropout      = 0.3,
    head_dropout      = 0.3,

    # Training
    epochs              = 20,
    batch_size          = 32,
    lr                  = 3e-4,
    weight_decay        = 2e-2,
    early_stop_patience = 4,
    focal_gamma         = 2.0,
    seed                = 42,
    push_to_hub         = True,
    load_from_hub       = False,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# =============================================================================
# M3 LABEL SCHEMAS
# =============================================================================
M3_LABEL_SCHEMAS: Dict = {
    "tempo_bpm":        (45.0,  170.0),
    "musical_valence":  (-1.0,   1.0),
    "tonality":         ["atonal", "major", "minor"],
    "harmonic_style":   ["atonal", "chromatic", "cluster",
                         "diatonic", "modal", "pentatonic", "whole_tone"],
    "dynamic_shape_m4": ["crescendo", "diminuendo", "flat",
                         "subito_forte", "subito_piano",
                         "sustained", "swell", "terraced"],
    "rhythm_style":     ["drive", "off", "ostinato", "pulse", "rubato", "sparse"],
    "texture":          ["ambient", "chamber", "full", "hybrid", "solo"],
    "orchestration":    ["ambient_pad", "brass", "choir", "electronic",
                         "ethnic", "guitar", "harp", "organ",
                         "percussion", "piano", "solo_voice",
                         "strings", "synth", "woodwinds"],
}

M3_REG_HEADS = ["tempo_bpm", "musical_valence"]
M3_CLS_HEADS = ["tonality", "harmonic_style",
                "dynamic_shape_m4", "rhythm_style", "texture"]
M3_ML_HEADS  = ["orchestration"]

M3_CLS_SIZES  = {k: len(M3_LABEL_SCHEMAS[k]) for k in M3_CLS_HEADS}
M3_ML_SIZES   = {k: len(M3_LABEL_SCHEMAS[k]) for k in M3_ML_HEADS}
M3_CLS_TO_IDX = {k: {v: i for i, v in enumerate(M3_LABEL_SCHEMAS[k])}
                 for k in M3_CLS_HEADS}
M3_IDX_TO_CLS = {k: {i: v for i, v in enumerate(M3_LABEL_SCHEMAS[k])}
                 for k in M3_CLS_HEADS}
M3_ML_TO_IDX  = {k: {v: i for i, v in enumerate(M3_LABEL_SCHEMAS[k])}
                 for k in M3_ML_HEADS}

M3_OPTIONAL_CLS = {"harmonic_style"}

# =============================================================================
# M1 LABEL SCHEMAS — needed to parse ground truth M1 labels from JSON
# =============================================================================
M1_CLS_FIELDS = {
    "emotional_valence":       {"Positive_Uplifting":0,"Neutral_Complex":1,
                                "Tension_Action":2,"Negative_Distressing":3},
    "conflict_nature":         {"Physical_Danger":0,"Psychological_Tension":1,
                                "Interpersonal_Conflict":2,"Moral_Dilemma":3,
                                "Environmental_Threat":4,"Unknown_Threat":5},
    "acoustic_space":          {"Interior_Small":0,"Interior_Large":1,
                                "Outdoor_Natural":2,"Outdoor_Urban":3,
                                "Vehicle":4,"Abstract":5},
    "reality_layer":           {"Present":0,"Memory":1,"Dream":2,
                                "Internal":3,"Distorted":4},
    "score_dynamic_shape":     {"Build_Release":0,"Sustained":1,
                                "Sudden_Drop":2,"Flat":3},
    "scene_interaction_tone":  {"Conflict":0,"Bonding":1,"Expository":2,
                                "Negotiation":3,"Reflective":4},
}
M1_CLS_DIMS = {
    "emotional_valence": 4, "conflict_nature": 6, "acoustic_space": 6,
    "reality_layer": 5, "score_dynamic_shape": 4, "scene_interaction_tone": 5,
}
M1_REG_FIELDS = {
    "pacing_intensity":  (1.0, 10.0),
    "action_intensity":  (0.0, 10.0),
    "scene_tension_raw": (1.0, 10.0),
    "scene_arousal":     (0.0,  1.0),
}
M1_REG_NORM_KEYS = ["pacing_norm", "action_norm", "tension_norm", "arousal_score"]
M1_ML_FIELDS = {
    "emotion_tags": {"Anger":0,"Joy":1,"Sadness":2,"Fear":3,
                     "Disgust":4,"Surprise":5,"Neutral":6}
}

# M2 label schemas needed for parsing ground truth from JSON
M2_LABEL_SCHEMAS: Dict = {
    "tension_level":          (1.0, 10.0),
    "arousal_level":          (1.0, 10.0),
    "narrative_arc_position": ["Setup","Rising","Climax","Falling","Resolution"],
    "foreshadowing_type":     ["None","Foreshadow"],
    "transition_type":        ["attacca","fade","segue","silence","cut"],
}
M2_ARC_TO_IDX   = {v: i for i, v in enumerate(M2_LABEL_SCHEMAS["narrative_arc_position"])}
M2_FSHAD_TO_IDX = {"None": 0, "Foreshadow": 1}
M2_TRANS_TO_IDX = {v: i for i, v in enumerate(M2_LABEL_SCHEMAS["transition_type"])}

# =============================================================================
# NORMALISATION
# =============================================================================
def m3_norm(value: float, field: str) -> float:
    lo, hi = M3_LABEL_SCHEMAS[field]
    return max(0.0, min(1.0, (float(value) - lo) / (hi - lo)))

def m3_denorm(norm_val: float, field: str) -> float:
    lo, hi = M3_LABEL_SCHEMAS[field]
    return float(norm_val) * (hi - lo) + lo

def _norm(v, lo, hi):
    return max(0.0, min(1.0, (float(v) - lo) / (hi - lo)))

# =============================================================================
# FOCAL LOSS
# =============================================================================
def focal_bce_loss(logits: torch.Tensor, targets: torch.Tensor,
                   gamma: float = 2.0) -> torch.Tensor:
    bce          = F.binary_cross_entropy_with_logits(
                       logits, targets, reduction="none")
    prob         = torch.sigmoid(logits)
    p_t          = targets * prob + (1 - targets) * (1 - prob)
    focal_weight = (1.0 - p_t) ** gamma
    return (focal_weight * bce).mean()

# =============================================================================
# M1 MODEL — replica for loading checkpoint
# =============================================================================
class M1ScenePerceptionModule(nn.Module):
    def __init__(self, backbone: str, embed_dim: int = 256, dropout: float = 0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone)
        h = self.encoder.config.hidden_size
        self.proj = nn.Sequential(
            nn.LayerNorm(h), nn.Linear(h, embed_dim),
            nn.GELU(), nn.Dropout(dropout),
        )
        E = embed_dim
        self.cls_heads = nn.ModuleDict({
            "emotional_valence":      nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 4)),
            "conflict_nature":        nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 6)),
            "acoustic_space":         nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 6)),
            "reality_layer":          nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 5)),
            "score_dynamic_shape":    nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 4)),
            "scene_interaction_tone": nn.Sequential(nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2, 5)),
        })
        self.reg_shared = nn.Sequential(nn.Linear(E, 64), nn.GELU(), nn.Dropout(dropout))
        self.reg_heads  = nn.ModuleDict({
            f: nn.Linear(64, 1) for f in
            ["pacing_intensity","action_intensity","scene_tension_raw","scene_arousal"]
        })
        self.ml_heads = nn.ModuleDict({
            "emotion_tags": nn.Sequential(
                nn.Linear(E,E//2), nn.GELU(), nn.Dropout(dropout), nn.Linear(E//2,7))
        })

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        emb = self.proj(cls)
        rb  = self.reg_shared(emb)
        return {
            "embedding":  emb,
            "cls_logits": {k: h(emb) for k, h in self.cls_heads.items()},
            "reg_preds":  {k: torch.sigmoid(h(rb)).squeeze(-1) for k, h in self.reg_heads.items()},
            "ml_logits":  {k: h(emb) for k, h in self.ml_heads.items()},
        }

# =============================================================================
# M2 MODEL — replica for loading checkpoint
# =============================================================================
class M2NarrativeContextModule(nn.Module):
    def __init__(self, scene_feat_dim=304, token_dim=256, n_heads=8,
                 n_layers=4, ffn_dim=512, tf_dropout=0.1,
                 proj_dropout=0.2, head_dropout=0.2, window_size=5):
        super().__init__()
        self.window_size = window_size
        self.scene_proj  = nn.Sequential(
            nn.Linear(scene_feat_dim, token_dim), nn.LayerNorm(token_dim),
            nn.GELU(), nn.Dropout(proj_dropout),
        )
        self.pos_embedding = nn.Embedding(window_size, token_dim)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=token_dim, nhead=n_heads, dim_feedforward=ffn_dim,
            dropout=tf_dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=n_layers, norm=nn.LayerNorm(token_dim))

        def _reg_head():
            return nn.Sequential(
                nn.Linear(token_dim, token_dim//2), nn.GELU(),
                nn.Dropout(head_dropout), nn.Linear(token_dim//2, 1))

        def _cls_head(n):
            return nn.Sequential(
                nn.Linear(token_dim, token_dim//2), nn.GELU(),
                nn.Dropout(head_dropout), nn.Linear(token_dim//2, n))

        # valence head REMOVED — matches current M2 checkpoint
        self.head_tension = _reg_head()
        self.head_arousal = _reg_head()
        self.head_shift   = nn.Sequential(
            nn.Linear(token_dim, token_dim//4), nn.GELU(),
            nn.Dropout(head_dropout), nn.Linear(token_dim//4, 1))
        self.head_arc   = _cls_head(5)
        self.head_fshad = _cls_head(4)   # matches Hub checkpoint (baseline M2)
        self.head_trans = _cls_head(5)

    def forward(self, window):
        B, W, _ = window.shape
        tokens   = self.scene_proj(window)
        pos_ids  = torch.arange(W, device=window.device).unsqueeze(0)
        tokens   = tokens + self.pos_embedding(pos_ids)
        pad_mask = (window.abs().sum(dim=-1) == 0)
        encoded  = self.transformer(tokens, src_key_padding_mask=pad_mask)
        ctx      = encoded[:, -1, :]
        return {
            "context_vector": ctx,
            "pred_tension":   torch.sigmoid(self.head_tension(ctx).squeeze(-1)),
            "pred_arousal":   torch.sigmoid(self.head_arousal(ctx).squeeze(-1)),
            "pred_shift":     self.head_shift(ctx).squeeze(-1),
            "pred_arc":       self.head_arc(ctx),
            "pred_fshad":     self.head_fshad(ctx),
            "pred_trans":     self.head_trans(ctx),
        }
# =============================================================================
# UPSTREAM FILE REGENERATION
# Downloads M1 + M2 checkpoints from Hub and runs inference if .pt missing
# =============================================================================
def _ensure_scene_vectors(cfg: dict) -> str:
    sv_path = cfg["scene_vectors"]
    if os.path.exists(sv_path):
        print(f"  ✓ Found scene_vectors.pt")
        return sv_path

    print("\n  scene_vectors.pt missing — downloading M1 from Hub + running inference")
    login(token=HF_READ_TOKEN, add_to_git_credential=False)
    m1_ckpt = hf_hub_download(
        repo_id=HF_M1_REPO, filename="m1_best.pt",
        token=HF_READ_TOKEN, local_dir=cfg["output_dir"])
    print(f"  ✓ Downloaded M1 checkpoint → {m1_ckpt}")

    tokenizer = AutoTokenizer.from_pretrained(cfg["m1_backbone"])
    model_m1  = M1ScenePerceptionModule(
        cfg["m1_backbone"], cfg["m1_embed_dim"], cfg["m1_dropout"]).to(device)
    ckpt = torch.load(m1_ckpt, map_location=device)
    model_m1.load_state_dict(ckpt["model_state"], strict=True)
    model_m1.eval()
    for p in model_m1.parameters():
        p.requires_grad = False

    _INT_EXT_RE   = re.compile(r'\b(INT|EXT|I/E|E/I)\b', re.IGNORECASE)
    _DAY_NIGHT_RE = re.compile(
        r'\b(DAY|NIGHT|DAWN|DUSK|CONTINUOUS|LATER|MORNING|EVENING)\b', re.IGNORECASE)
    _DAY_SYN   = {"DAY","DAWN","MORNING"}
    _NIGHT_SYN = {"NIGHT","DUSK","EVENING"}

    def _parse_header(h):
        ie = _INT_EXT_RE.search(h or "")
        dn = _DAY_NIGHT_RE.search(h or "")
        int_ext = (0 if ie.group(1).upper()=="INT" else 1) if ie else 2
        if dn:
            t = dn.group(1).upper()
            day_night = 0 if t in _DAY_SYN else (1 if t in _NIGHT_SYN else 2)
        else:
            day_night = 2
        return int_ext, day_night

    M1_OPT = {"conflict_nature","reality_layer","score_dynamic_shape","emotional_valence"}
    M1_REG_RANGES = {
        "pacing_intensity":(1.0,10.0),"action_intensity":(0.0,10.0),
        "scene_tension_raw":(1.0,10.0),"scene_arousal":(0.0,1.0),
    }

    json_files = sorted(
        glob.glob(os.path.join(cfg["data_dir"],"**/*.json"), recursive=True)
        or glob.glob(os.path.join(cfg["data_dir"],"*.json")))
    print(f"  Found {len(json_files)} JSON files")

    samples, skipped = [], 0
    for film_id, fpath in enumerate(json_files):
        with open(fpath,"r",encoding="utf-8") as f:
            data = json.load(f)
        scenes       = data.get("annotations",[]) if isinstance(data,dict) else data
        total_scenes = data.get("total_scenes",len(scenes)) if isinstance(data,dict) else len(scenes)
        for sc in scenes:
            text = sc.get("scene_text","").strip()
            if not text:
                skipped += 1; continue
            try:
                cls_l = {}
                for field, idx_map in M1_CLS_FIELDS.items():
                    v = sc.get(field)
                    if v not in idx_map:
                        if field in M1_OPT:
                            cls_l[field] = -1; continue
                        raise ValueError
                    cls_l[field] = idx_map[v]
                reg_l = {}
                for field,(lo,hi) in M1_REG_RANGES.items():
                    v = sc.get(field)
                    if v is None: raise ValueError
                    reg_l[field] = _norm(v,lo,hi)
                ml_v = torch.zeros(7)
                for tag in (sc.get("emotion_tags") or []):
                    i = M1_ML_FIELDS["emotion_tags"].get(
                        tag.strip() if isinstance(tag,str) else tag)
                    if i is not None: ml_v[i] = 1.0
                sid    = int(sc.get("scene_id",0))
                ie, dn = _parse_header(sc.get("scene_header",""))
                samples.append({
                    "text":text,"scene_id":sid,"film_id":film_id,
                    "position":sid/max(total_scenes,1),
                    "int_ext":ie,"day_night":dn,
                })
            except:
                skipped += 1

    print(f"  Parsed {len(samples)} scenes | skipped {skipped}")

    M1_CLS_LIST = list(M1_CLS_FIELDS.keys())
    M1_REG_LIST = list(M1_REG_RANGES.keys())
    A: Dict[str,list] = {
        "vecs":[],"scene_id":[],"film_id":[],"position":[],
        "int_ext":[],"day_night":[],"emo_probs":[],"dom_emo":[],
        **{f:[] for f in M1_CLS_LIST},
        **{f:[] for f in M1_REG_LIST},
    }

    with torch.no_grad():
        for i in range(0, len(samples), 64):
            batch_s = samples[i:i+64]
            enc = tokenizer(
                [s["text"] for s in batch_s],
                max_length=cfg["m1_max_length"],
                truncation=True, padding="max_length", return_tensors="pt")
            out = model_m1(enc["input_ids"].to(device),
                           enc["attention_mask"].to(device))
            A["vecs"].append(out["embedding"].cpu())
            for field in M1_CLS_LIST:
                A[field].append(out["cls_logits"][field].cpu().argmax(-1))
            for field in M1_REG_LIST:
                A[field].append(out["reg_preds"][field].cpu())
            ep = torch.sigmoid(out["ml_logits"]["emotion_tags"]).cpu()
            A["emo_probs"].append(ep)
            A["dom_emo"].append(ep.argmax(-1))
            A["scene_id"].append(torch.tensor([s["scene_id"] for s in batch_s],dtype=torch.long))
            A["film_id"].append( torch.tensor([s["film_id"]  for s in batch_s],dtype=torch.long))
            A["position"].append(torch.tensor([s["position"] for s in batch_s],dtype=torch.float32))
            A["int_ext"].append( torch.tensor([s["int_ext"]  for s in batch_s],dtype=torch.long))
            A["day_night"].append(torch.tensor([s["day_night"] for s in batch_s],dtype=torch.long))

    C = {k: torch.cat(v) for k,v in A.items()}
    torch.save({
        "scene_vector":           C["vecs"],
        "emotional_valence":      C["emotional_valence"],
        "conflict_nature":        C["conflict_nature"],
        "acoustic_space":         C["acoustic_space"],
        "reality_layer":          C["reality_layer"],
        "score_dynamic_shape":    C["score_dynamic_shape"],
        "scene_interaction_tone": C["scene_interaction_tone"],
        "pacing_norm":            C["pacing_intensity"],
        "action_norm":            C["action_intensity"],
        "tension_norm":           C["scene_tension_raw"],
        "arousal_score":          C["scene_arousal"],
        "emotion_tags_probs":     C["emo_probs"],
        "dominant_emotion":       C["dom_emo"],
        "scene_id":               C["scene_id"],
        "film_id":                C["film_id"],
        "position_in_film":       C["position"],
        "int_ext":                C["int_ext"],
        "day_night":              C["day_night"],
    }, sv_path)
    print(f"  ✓ scene_vectors.pt saved → {sv_path}")
    return sv_path


def _ensure_narrative_vectors(cfg: dict, scene_vecs: dict) -> dict:
    all_present = all(
        os.path.exists(cfg[f"narrative_vectors_{s}"])
        for s in ["train","val","test"])

    if all_present:
        print("  ✓ Found all narrative_vectors split files")
    else:
        print("\n  narrative_vectors missing — downloading M2 from Hub + running inference")
        login(token=HF_READ_TOKEN, add_to_git_credential=False)
        m2_ckpt = hf_hub_download(
            repo_id=HF_M2_REPO, filename="m2_best.pt",
            token=HF_READ_TOKEN, local_dir=cfg["output_dir"])
        print(f"  ✓ Downloaded M2 checkpoint → {m2_ckpt}")

        # Build scene feature vectors (304-d) same as M2 pipeline
        M1_INT_EXT_DIM   = 3
        M1_DAY_NIGHT_DIM = 3
        _SCENE_FEAT_DIM  = 304

        M1_CLS_DIMS_ordered = {
            "emotional_valence":4,"conflict_nature":6,"acoustic_space":6,
            "reality_layer":5,"score_dynamic_shape":4,"scene_interaction_tone":5,
        }

        def build_scene_feature(sv, idx):
            parts = [sv["scene_vector"][idx]]
            for field, dim in M1_CLS_DIMS_ordered.items():
                oh  = torch.zeros(dim)
                val = sv[field][idx].item()
                if 0 <= val < dim: oh[val] = 1.0
                parts.append(oh)
            parts += [
                sv["pacing_norm"][idx].unsqueeze(0),
                sv["action_norm"][idx].unsqueeze(0),
                sv["tension_norm"][idx].unsqueeze(0),
                sv["arousal_score"][idx].unsqueeze(0),
            ]
            parts.append(sv["emotion_tags_probs"][idx])
            parts.append(sv["position_in_film"][idx].unsqueeze(0))
            ie_oh = torch.zeros(M1_INT_EXT_DIM)
            ie_v  = sv["int_ext"][idx].item()
            if 0 <= ie_v < M1_INT_EXT_DIM: ie_oh[ie_v] = 1.0
            parts.append(ie_oh)
            dn_oh = torch.zeros(M1_DAY_NIGHT_DIM)
            dn_v  = sv["day_night"][idx].item()
            if 0 <= dn_v < M1_DAY_NIGHT_DIM: dn_oh[dn_v] = 1.0
            parts.append(dn_oh)
            return torch.cat(parts, dim=0)

        # Load M2 model
        model_m2 = M2NarrativeContextModule(
            scene_feat_dim = cfg["m2_scene_feat_dim"],
            token_dim      = cfg["m2_token_dim"],
            n_heads        = cfg["m2_n_heads"],
            n_layers       = cfg["m2_n_layers"],
            ffn_dim        = cfg["m2_ffn_dim"],
            tf_dropout     = cfg["m2_tf_dropout"],
            proj_dropout   = cfg["m2_proj_dropout"],
            head_dropout   = cfg["m2_head_dropout"],
            window_size    = cfg["m2_window_size"],
        ).to(device)
        ckpt_m2 = torch.load(m2_ckpt, map_location=device)
        model_m2.load_state_dict(ckpt_m2["model_state"], strict=True)
        model_m2.eval()
        for p in model_m2.parameters():
            p.requires_grad = False

        # Build ordered scene list per film
        film_ids  = scene_vecs["film_id"]
        scene_ids = scene_vecs["scene_id"]
        film_to_indices: Dict[int, List[Tuple[int,int]]] = {}
        for gi in range(len(film_ids)):
            fid = film_ids[gi].item()
            sid = scene_ids[gi].item()
            film_to_indices.setdefault(fid, []).append((gi, sid))
        for fid in film_to_indices:
            film_to_indices[fid].sort(key=lambda x: x[1])

        W_SIZE = cfg["m2_window_size"]
        lo_t, hi_t = 1.0, 10.0
        lo_a, hi_a = 1.0, 10.0

        A: Dict[str,list] = {k:[] for k in [
            "ctx","tension","arousal","shift",
            "arc","fshad","trans","scene_id","film_id","position"]}

        with torch.no_grad():
            for fid, ordered in film_to_indices.items():
                for pos,(gi,sid) in enumerate(ordered):
                    win_idxs = []
                    for offset in range(-(W_SIZE-1), 1):
                        wp = pos + offset
                        win_idxs.append(None if wp < 0 else ordered[wp][0])
                    pad  = torch.zeros(_SCENE_FEAT_DIM)
                    feats = [pad if wi is None else build_scene_feature(scene_vecs, wi)
                             for wi in win_idxs]
                    window = torch.stack(feats, dim=0).unsqueeze(0).to(device)
                    out    = model_m2(window)

                    A["ctx"].append(out["context_vector"].cpu())
                    A["tension"].append(
                        out["pred_tension"].cpu() * (hi_t - lo_t) + lo_t)
                    A["arousal"].append(
                        out["pred_arousal"].cpu() * (hi_a - lo_a) + lo_a)
                    A["shift"].append(
                        (torch.sigmoid(out["pred_shift"].cpu()) > 0.5).long())
                    A["arc"].append(  out["pred_arc"].cpu().argmax(-1))
                    A["fshad"].append(out["pred_fshad"].cpu().argmax(-1))
                    A["trans"].append(out["pred_trans"].cpu().argmax(-1))
                    A["scene_id"].append(scene_ids[gi].unsqueeze(0))
                    A["film_id"].append( film_ids[gi].unsqueeze(0))
                    A["position"].append(
                        scene_vecs["position_in_film"][gi].unsqueeze(0))

        C = {k: torch.cat(v) for k,v in A.items()}
        N = C["ctx"].shape[0]

        # Stratified split same as M2 — on arc position
        # Film-level split to match M2 training exactly
        from collections import defaultdict as _dd
        _film_to_idx: Dict[int, List[int]] = _dd(list)
        for i in range(N):
            fid = C["film_id"][i].item()
            _film_to_idx[fid].append(i)

        _all_films = sorted(_film_to_idx.keys())
        _rng       = random.Random(42)
        _rng.shuffle(_all_films)

        _n_films = len(_all_films)
        _n_test  = max(1, round(0.10 * _n_films))
        _n_val   = max(1, round(0.10 * _n_films))
        _n_train = _n_films - _n_val - _n_test

        tr_idx = [i for f in _all_films[:_n_train]
                  for i in _film_to_idx[f]]
        val_idx = [i for f in _all_films[_n_train:_n_train+_n_val]
                   for i in _film_to_idx[f]]
        te_idx  = [i for f in _all_films[_n_train+_n_val:]
                   for i in _film_to_idx[f]]

        for split_name, indices in [("train",tr_idx),("val",val_idx),("test",te_idx)]:
            sub = {k: C[k][indices] for k in C}
            out_path = cfg[f"narrative_vectors_{split_name}"]
            torch.save({
                "context_vector":          sub["ctx"],
                "tension_level":           sub["tension"],
                "arousal_level":           sub["arousal"],
                "emotional_shift_trigger": sub["shift"],
                "narrative_arc_position":  sub["arc"],
                "foreshadowing_type":      sub["fshad"],
                "transition_type":         sub["trans"],
                "scene_id":                sub["scene_id"],
                "film_id":                 sub["film_id"],
                "position_in_film":        sub["position"],
                "split":                   split_name,
            }, out_path)
            print(f"  ✓ {split_name:>5}: {len(indices)} vectors → {out_path}")

    # Merge all three splits
    parts = []
    for s in ["train","val","test"]:
        parts.append(torch.load(cfg[f"narrative_vectors_{s}"], map_location="cpu"))
    tensor_keys = [k for k in parts[0] if isinstance(parts[0][k], torch.Tensor)]
    merged = {k: torch.cat([p[k] for p in parts], dim=0) for k in tensor_keys}
    print(f"  ✓ Merged: {merged['context_vector'].shape[0]} total narrative vectors")
    return merged

# =============================================================================
# INPUT FEATURE BUILDER
# Builds the 313-d input vector from ground truth JSON labels
# =============================================================================
def build_m3_input(sc: dict,
                   context_vec: torch.Tensor) -> Optional[torch.Tensor]:
    """
    313-d = 41 (M1 GT labels) + 16 (M2 GT labels) + 256 (context_vector)
    During training: uses ground truth labels from JSON
    During inference: uses predicted labels from scene_vectors + narrative_vectors
    """
    try:
        parts = []

        # ── M1 labels (41-d) ─────────────────────────────────────────────────
        for field, idx_map in M1_CLS_FIELDS.items():
            dim = M1_CLS_DIMS[field]
            oh  = torch.zeros(dim)
            v   = sc.get(field)
            idx = idx_map.get(v)
            if idx is not None:
                oh[idx] = 1.0
            parts.append(oh)

        # Regression scalars — normalised 0-1
        reg_map = {
            "pacing_intensity":  (1.0,10.0),
            "action_intensity":  (0.0,10.0),
            "scene_tension_raw": (1.0,10.0),
            "scene_arousal":     (0.0, 1.0),
        }
        for field,(lo,hi) in reg_map.items():
            v = sc.get(field)
            val = _norm(v,lo,hi) if v is not None else 0.0
            parts.append(torch.tensor([val], dtype=torch.float32))

        # Emotion tags (7-hot)
        em_vec = torch.zeros(7)
        raw_tags = sc.get("emotion_tags") or []
        if isinstance(raw_tags, str):
            raw_tags = [t.strip() for t in raw_tags.split(",")]
        for tag in raw_tags:
            i = M1_ML_FIELDS["emotion_tags"].get(tag)
            if i is not None: em_vec[i] = 1.0
        parts.append(em_vec)

        # ── M2 labels (16-d) ─────────────────────────────────────────────────
        # tension_level scalar (normalised)
        tl = sc.get("tension_level")
        parts.append(torch.tensor(
            [_norm(tl,1.0,10.0)] if tl is not None else [0.5],
            dtype=torch.float32))

        # arousal_level scalar (normalised)
        al = sc.get("arousal_level")
        parts.append(torch.tensor(
            [_norm(al,1.0,10.0)] if al is not None else [0.5],
            dtype=torch.float32))

        # emotional_shift_trigger scalar
        raw_shift = sc.get("emotional_shift_trigger")
        if isinstance(raw_shift, bool):
            shift_val = float(raw_shift)
        elif isinstance(raw_shift, str):
            shift_val = 1.0 if raw_shift.strip().lower() in ("true","1","yes") else 0.0
        elif raw_shift is not None:
            shift_val = float(bool(raw_shift))
        else:
            shift_val = 0.0
        parts.append(torch.tensor([shift_val], dtype=torch.float32))

        # narrative_arc_position one-hot (5)
        arc_oh = torch.zeros(5)
        arc_v  = M2_ARC_TO_IDX.get(sc.get("narrative_arc_position"))
        if arc_v is not None: arc_oh[arc_v] = 1.0
        parts.append(arc_oh)

        # foreshadowing_type one-hot (4) — matches baseline M2 Hub checkpoint
        fsh_oh = torch.zeros(4)
        fsh_map = {"None":0,"Foreshadow":1,"Payoff":2,"Echo":3}
        fsh_v   = sc.get("foreshadowing_type")
        fsh_i   = fsh_map.get(fsh_v, 0)
        fsh_oh[fsh_i] = 1.0
        parts.append(fsh_oh)

        # transition_type one-hot (5)
        tra_oh = torch.zeros(5)
        tra_v  = M2_TRANS_TO_IDX.get(sc.get("transition_type"))
        if tra_v is not None: tra_oh[tra_v] = 1.0
        parts.append(tra_oh)

        # ── context_vector (256-d) ────────────────────────────────────────────
        parts.append(context_vec)

        combined = torch.cat(parts, dim=0)
        assert combined.shape[0] == 314, \
            f"Input dim mismatch: {combined.shape[0]} != 314"
        return combined

    except Exception:
        return None

# =============================================================================
# DATASET
# =============================================================================
class MusicDataset(Dataset):
    def __init__(self, data_dir: str, narrative_vecs: dict):
        self.samples: List[dict] = []

        # Build lookup: (film_id, scene_id) → index in narrative_vecs
        nv_index: Dict[Tuple[int,int], int] = {}
        for gi in range(len(narrative_vecs["scene_id"])):
            fid = narrative_vecs["film_id"][gi].item()
            sid = narrative_vecs["scene_id"][gi].item()
            nv_index[(fid, sid)] = gi

        json_files = sorted(
            glob.glob(os.path.join(data_dir,"**/*.json"), recursive=True)
            or glob.glob(os.path.join(data_dir,"*.json")))
        print(f"  Found {len(json_files)} JSON files for M3 labels")

        skipped = 0
        for film_id, fpath in enumerate(json_files):
            with open(fpath,"r",encoding="utf-8") as f:
                data = json.load(f)
            scenes = data.get("annotations",[]) if isinstance(data,dict) else data

            for sc in scenes:
                scene_id = int(sc.get("scene_id",0))
                key      = (film_id, scene_id)
                nv_gi    = nv_index.get(key)
                if nv_gi is None:
                    skipped += 1; continue

                # Get context_vector for this scene
                context_vec = narrative_vecs["context_vector"][nv_gi]

                # Build 313-d input from ground truth labels
                combined = build_m3_input(sc, context_vec)
                if combined is None:
                    skipped += 1; continue

                # Parse M3 output labels
                labels = self._parse_labels(sc)
                if labels is None:
                    skipped += 1; continue

                self.samples.append({
                    "combined": combined,
                    "labels":   labels,
                    "scene_id": scene_id,
                    "film_id":  film_id,
                    "position": narrative_vecs["position_in_film"][nv_gi].item(),
                    "raw_text": sc.get("scene_text","")[:400],
                    "raw_labels": {
                        "tempo_bpm":        sc.get("tempo_bpm"),
                        "musical_valence":  sc.get("musical_valence"),
                        "tonality":         sc.get("tonality"),
                        "harmonic_style":   sc.get("harmonic_style"),
                        "dynamic_shape_m4": sc.get("dynamic_shape_m4"),
                        "rhythm_style":     sc.get("rhythm_style"),
                        "texture":          sc.get("texture"),
                        "orchestration":    sc.get("orchestration",[]),
                    },
                })

        print(f"  Built {len(self.samples)} samples | skipped {skipped}")
        self._nv = narrative_vecs

    def _parse_labels(self, sc: dict) -> Optional[dict]:
        try:
            labels = {}
            for field in M3_REG_HEADS:
                v = sc.get(field)
                if v is None: return None
                labels[field] = m3_norm(float(v), field)
            for field in M3_CLS_HEADS:
                v   = sc.get(field)
                idm = M3_CLS_TO_IDX[field]
                if v not in idm:
                    if field in M3_OPTIONAL_CLS:
                        labels[field] = -1; continue
                    return None
                labels[field] = idm[v]
            raw = sc.get("orchestration",[])
            if isinstance(raw, str):
                raw = [x.strip() for x in raw.split(",")]
            vec = torch.zeros(M3_ML_SIZES["orchestration"])
            for inst in raw:
                idx = M3_ML_TO_IDX["orchestration"].get(inst)
                if idx is not None: vec[idx] = 1.0
            labels["orchestration"] = vec
            return labels
        except (TypeError, ValueError, KeyError):
            return None

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict:
        s = self.samples[idx]
        return {
            "combined":   s["combined"],
            "reg_labels": {k: torch.tensor(s["labels"][k], dtype=torch.float32)
                           for k in M3_REG_HEADS},
            "cls_labels": {k: torch.tensor(s["labels"][k], dtype=torch.long)
                           for k in M3_CLS_HEADS},
            "ml_labels":  {"orchestration": s["labels"]["orchestration"]},
            "scene_id":   torch.tensor(s["scene_id"],  dtype=torch.long),
            "film_id":    torch.tensor(s["film_id"],   dtype=torch.long),
            "position":   torch.tensor(s["position"],  dtype=torch.float32),
            "raw_text":   s["raw_text"],
            "raw_labels": s["raw_labels"],
        }


def collate_fn(batch: List[dict]) -> dict:
    return {
        "combined":   torch.stack([b["combined"]  for b in batch]),
        "reg_labels": {k: torch.stack([b["reg_labels"][k] for b in batch])
                       for k in M3_REG_HEADS},
        "cls_labels": {k: torch.stack([b["cls_labels"][k] for b in batch])
                       for k in M3_CLS_HEADS},
        "ml_labels":  {"orchestration":
                       torch.stack([b["ml_labels"]["orchestration"] for b in batch])},
        "scene_id":   torch.stack([b["scene_id"]  for b in batch]),
        "film_id":    torch.stack([b["film_id"]   for b in batch]),
        "position":   torch.stack([b["position"]  for b in batch]),
        "raw_text":   [b["raw_text"]   for b in batch],
        "raw_labels": [b["raw_labels"] for b in batch],
    }

# =============================================================================
# MODEL
# =============================================================================
class MusicDescriptorModule(nn.Module):
    """
    Input: 313-d (41 M1 labels + 16 M2 labels + 256 context_vector)
    → Linear(313→256) + LayerNorm + GELU + Dropout → music_token (256-d)
    → 8 task heads
    """
    def __init__(self, input_dim=313, proj_dim=256, head_hidden=128,
                 proj_dropout=0.3, head_dropout=0.3):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(proj_dropout),
        )

        def _reg_head():
            return nn.Sequential(
                nn.Linear(proj_dim, head_hidden), nn.GELU(),
                nn.Dropout(head_dropout), nn.Linear(head_hidden, 1))

        def _cls_head(n):
            return nn.Sequential(
                nn.Linear(proj_dim, head_hidden), nn.GELU(),
                nn.Dropout(head_dropout), nn.Linear(head_hidden, n))

        self.head_tempo    = _reg_head()
        self.head_valence  = _reg_head()
        self.head_tonality = _cls_head(M3_CLS_SIZES["tonality"])
        self.head_harmonic = _cls_head(M3_CLS_SIZES["harmonic_style"])
        self.head_dynamic  = _cls_head(M3_CLS_SIZES["dynamic_shape_m4"])
        self.head_rhythm   = _cls_head(M3_CLS_SIZES["rhythm_style"])
        self.head_texture  = _cls_head(M3_CLS_SIZES["texture"])
        self.head_orch     = nn.Sequential(
            nn.Linear(proj_dim, head_hidden), nn.GELU(),
            nn.Dropout(head_dropout),
            nn.Linear(head_hidden, M3_ML_SIZES["orchestration"]))

    def forward(self, x: torch.Tensor) -> dict:
        z = self.proj(x)
        return {
            "music_vector":     z,
            "pred_tempo":       torch.sigmoid(self.head_tempo(z).squeeze(-1)),
            "pred_valence":     torch.tanh(   self.head_valence(z).squeeze(-1)),
            "pred_tonality":    self.head_tonality(z),
            "pred_harmonic":    self.head_harmonic(z),
            "pred_dynamic":     self.head_dynamic(z),
            "pred_rhythm":      self.head_rhythm(z),
            "pred_texture":     self.head_texture(z),
            "pred_orch_logits": self.head_orch(z),
        }

# =============================================================================
# LOSS WEIGHTS
# =============================================================================
LOSS_W = {
    "tempo_bpm":        1.2,
    "musical_valence":  1.5,
    "tonality":         1.5,
    "harmonic_style":   1.2,
    "dynamic_shape_m4": 1.0,
    "rhythm_style":     1.2,
    "texture":          1.0,
    "orchestration":    1.8,
}


def compute_loss(outputs: dict, batch: dict,
                 focal_gamma: float) -> Tuple[torch.Tensor, dict]:
    bkd   = {}
    total = torch.zeros(1, device=device)

    l = F.mse_loss(outputs["pred_tempo"],
                   batch["reg_labels"]["tempo_bpm"].to(device))
    total = total + l * LOSS_W["tempo_bpm"]
    bkd["reg/tempo"] = l.item()

    vp = (outputs["pred_valence"] + 1.0) / 2.0
    vt = batch["reg_labels"]["musical_valence"].to(device)
    l  = F.mse_loss(vp, vt)
    total = total + l * LOSS_W["musical_valence"]
    bkd["reg/valence"] = l.item()

    for field, pk in [
        ("tonality",         "pred_tonality"),
        ("harmonic_style",   "pred_harmonic"),
        ("dynamic_shape_m4", "pred_dynamic"),
        ("rhythm_style",     "pred_rhythm"),
        ("texture",          "pred_texture"),
    ]:
        logits = outputs[pk]
        tgt    = batch["cls_labels"][field].to(device)
        mask   = tgt >= 0
        if mask.sum() == 0: continue
        l = F.cross_entropy(logits[mask], tgt[mask])
        total = total + l * LOSS_W[field]
        bkd[f"cls/{field}"] = l.item()

    l = focal_bce_loss(
        outputs["pred_orch_logits"],
        batch["ml_labels"]["orchestration"].to(device),
        gamma=focal_gamma)
    total = total + l * LOSS_W["orchestration"]
    bkd["ml/orchestration"] = l.item()

    bkd["total"] = total.item()
    return total, bkd

# =============================================================================
# METRICS
# =============================================================================
def compute_metrics(all_out: list, all_bat: list) -> list:
    tmp_p, tmp_t = [], []
    val_p, val_t = [], []
    cls_p = {k: [] for k in M3_CLS_HEADS}
    cls_t = {k: [] for k in M3_CLS_HEADS}
    orc_p, orc_t = [], []

    lo_bpm, hi_bpm = M3_LABEL_SCHEMAS["tempo_bpm"]
    lo_val, hi_val = M3_LABEL_SCHEMAS["musical_valence"]

    for o, b in zip(all_out, all_bat):
        tp = o["pred_tempo"].numpy() * (hi_bpm - lo_bpm) + lo_bpm
        tt = b["reg_labels"]["tempo_bpm"].numpy() * (hi_bpm - lo_bpm) + lo_bpm
        tmp_p.extend(tp.tolist()); tmp_t.extend(tt.tolist())

        vp = o["pred_valence"].numpy() * (hi_val - lo_val) / 2.0 + (hi_val + lo_val) / 2.0
        vt = b["reg_labels"]["musical_valence"].numpy() * (hi_val - lo_val) + lo_val
        val_p.extend(vp.tolist()); val_t.extend(vt.tolist())

        for field, pk in [
            ("tonality",         "pred_tonality"),
            ("harmonic_style",   "pred_harmonic"),
            ("dynamic_shape_m4", "pred_dynamic"),
            ("rhythm_style",     "pred_rhythm"),
            ("texture",          "pred_texture"),
        ]:
            tgt  = b["cls_labels"][field].numpy()
            mask = tgt >= 0
            if mask.sum() == 0: continue
            cls_p[field].extend(o[pk][mask].argmax(-1).numpy().tolist())
            cls_t[field].extend(tgt[mask].tolist())

        pp = (torch.sigmoid(o["pred_orch_logits"]) > 0.5).int().numpy()
        tt = b["ml_labels"]["orchestration"].int().numpy()
        orc_p.append(pp); orc_t.append(tt)

    rows = []
    for label, pp, tt in [("tempo_bpm",tmp_p,tmp_t),("musical_valence",val_p,val_t)]:
        rows.append({
            "field": label, "type": "REG",
            "acc": None, "f1m": None, "f1w": None,
            "mae": mean_absolute_error(tt,pp),
            "r2":  r2_score(tt,pp),
            "prec": None, "rec": None,
        })
    for field in M3_CLS_HEADS:
        if not cls_t[field]: continue
        rows.append({
            "field": field, "type": "CLS",
            "acc": accuracy_score(cls_t[field], cls_p[field]),
            "f1m": f1_score(cls_t[field], cls_p[field], average="macro",    zero_division=0),
            "f1w": f1_score(cls_t[field], cls_p[field], average="weighted", zero_division=0),
            "mae": None, "r2": None, "prec": None, "rec": None,
        })
    if orc_p:
        p = np.vstack(orc_p); t = np.vstack(orc_t)
        rows.append({
            "field": "orchestration", "type": "ML",
            "acc": None,
            "f1m": f1_score(t,p,average="macro",    zero_division=0),
            "f1w": f1_score(t,p,average="weighted", zero_division=0),
            "mae": None, "r2": None, "prec": None, "rec": None,
        })
    return rows

# =============================================================================
# METRICS TABLE PRINTER
# =============================================================================
def print_metrics_table(epoch, n_epochs, avg_train, avg_val,
                        rows, best_val, patience_counter, patience):
    W  = 110
    EQ = "═" * W
    DH = "─" * W
    nb = "  ★ NEW BEST" if avg_val <= best_val else \
         f"  (no improvement {patience_counter}/{patience})"
    print(f"\n{EQ}")
    print(f"  EPOCH {epoch:>2}/{n_epochs}"
          f"   │   Train Loss: {avg_train:.4f}"
          f"   │   Val Loss: {avg_val:.4f}"
          f"   │   Best: {min(best_val,avg_val):.4f}{nb}")
    print(EQ)
    print(f"  {'Head':<24} {'Type':<5} "
          f"{'Accuracy':>10} {'F1-Macro':>10} {'F1-Wtd':>9} "
          f"{'Prec':>8} {'Recall':>8} {'MAE':>10} {'R²':>9}")
    print(DH)
    for r in rows:
        acc  = f"{r['acc']:.4f}"  if r['acc']  is not None else "    —    "
        f1m  = f"{r['f1m']:.4f}"  if r['f1m']  is not None else "    —    "
        f1w  = f"{r['f1w']:.4f}"  if r['f1w']  is not None else "    —    "
        prec = f"{r['prec']:.4f}" if r['prec'] is not None else "   —    "
        rec  = f"{r['rec']:.4f}"  if r['rec']  is not None else "   —    "
        mae  = f"{r['mae']:.4f}"  if r['mae']  is not None else "    —    "
        r2   = f"{r['r2']:.4f}"   if r['r2']   is not None else "    —    "
        print(f"  {r['field']:<24} {r['type']:<5} "
              f"{acc:>10} {f1m:>10} {f1w:>9} "
              f"{prec:>8} {rec:>8} {mae:>10} {r2:>9}")
    print(DH)

# =============================================================================
# PER-EPOCH SCENE PREVIEW
# =============================================================================
def print_scene_preview(model: nn.Module, val_ds, epoch: int):
    model.eval()
    idx    = random.randint(0, len(val_ds) - 1)
    sample = val_ds[idx]
    s      = val_ds.dataset.samples[val_ds.indices[idx]]

    with torch.no_grad():
        out = model(sample["combined"].unsqueeze(0).to(device))

    lo_bpm, hi_bpm = M3_LABEL_SCHEMAS["tempo_bpm"]
    lo_val, hi_val = M3_LABEL_SCHEMAS["musical_valence"]

    pred_tempo   = out["pred_tempo"].item()   * (hi_bpm - lo_bpm) + lo_bpm
    pred_valence = out["pred_valence"].item() * (hi_val - lo_val) / 2.0 \
                   + (hi_val + lo_val) / 2.0
    pred_ton  = M3_IDX_TO_CLS["tonality"][        out["pred_tonality"].argmax(-1).item()]
    pred_harm = M3_IDX_TO_CLS["harmonic_style"][  out["pred_harmonic"].argmax(-1).item()]
    pred_dyn  = M3_IDX_TO_CLS["dynamic_shape_m4"][out["pred_dynamic"].argmax(-1).item()]
    pred_rhy  = M3_IDX_TO_CLS["rhythm_style"][    out["pred_rhythm"].argmax(-1).item()]
    pred_tex  = M3_IDX_TO_CLS["texture"][         out["pred_texture"].argmax(-1).item()]

    orch_probs  = torch.sigmoid(out["pred_orch_logits"]).squeeze(0)
    orch_labels = M3_LABEL_SCHEMAS["orchestration"]
    pred_orch   = [orch_labels[i] for i,p in enumerate(orch_probs)
                   if p.item() > 0.5] or ["(none)"]

    raw = s["raw_labels"]
    W   = 92
    EQ  = "═" * W
    DH  = "─" * W

    print(f"\n{EQ}")
    print(f"  🎵  SCENE PREVIEW  —  Epoch {epoch}"
          f"  (val #{idx} | film {s['film_id']}"
          f" | scene {s['scene_id']}"
          f" | pos={s['position']:.2f})")
    print(EQ)
    for line in textwrap.fill(s["raw_text"], width=W-4).split("\n"):
        print(f"  {line}")
    print(DH)
    print(f"  {'Head':<24} {'TRUE':<22} {'PREDICTED':<22} {'Notes'}")
    print(DH)

    def _tv(key):
        v = raw.get(key)
        return str(v) if v is not None else "?"

    def _chk(tv, pv):
        return "✓" if str(tv).strip().lower() == str(pv).strip().lower() else "✗"

    def _err(tv, pv_f):
        try:    return f"Δ={abs(float(tv)-pv_f):.2f}"
        except: return ""

    for label, pv_f in [("tempo_bpm",pred_tempo),("musical_valence",pred_valence)]:
        tv = _tv(label)
        print(f"  {label:<24} {tv:<22} {pv_f:<22.3f} {_err(tv,pv_f)}")
    print(DH)
    for label, pv in [
        ("tonality",pred_ton),("harmonic_style",pred_harm),
        ("dynamic_shape_m4",pred_dyn),("rhythm_style",pred_rhy),("texture",pred_tex)]:
        tv = _tv(label)
        print(f"  {label:<24} {tv:<22} {pv:<22} {_chk(tv,pv)}")
    print(DH)
    true_orch = raw.get("orchestration",[])
    if isinstance(true_orch,str):
        true_orch = [x.strip() for x in true_orch.split(",")]
    print(f"  {'orchestration':<24} {', '.join(true_orch) if true_orch else '(none)'}")
    print(f"  {'':24} predicted: {', '.join(pred_orch)}")
    print(f"{EQ}\n")
    model.train()

# =============================================================================
# HUGGINGFACE HELPERS
# =============================================================================
def push_m3_to_hub(cfg: dict, ckpt_path: str):
    if not os.path.exists(ckpt_path):
        print(f"  ⚠  Checkpoint not found — skipping Hub push")
        return
    print("\n── Pushing M3 checkpoint to HuggingFace Hub ────────────────────")
    try:
        login(token=HF_WRITE_TOKEN, add_to_git_credential=False)
        api = HfApi()
        api.create_repo(repo_id=HF_M3_REPO, repo_type="model",
                        exist_ok=True, token=HF_WRITE_TOKEN, private=False)
        api.upload_file(
            path_or_fileobj=ckpt_path, path_in_repo="m3_best.pt",
            repo_id=HF_M3_REPO, repo_type="model", token=HF_WRITE_TOKEN,
            commit_message="Upload best M3 checkpoint — 313-d input with GT labels",
        )
        print("  ✓ Uploaded m3_best.pt")
        card = f"""---
language: en
tags: [music-description, film-scoring, multi-task, pytorch]
---
# Music Descriptor Module 3

Input: 313-d = 41-d M1 GT labels + 16-d M2 GT labels + 256-d M2 context_vector.
Final module in the Scene → Narrative → Music pipeline.

## 8 Music Descriptor Heads
| # | Head | Type | Output |
|---|------|------|--------|
| 1 | tempo_bpm | regression | 45–170 BPM |
| 2 | musical_valence | regression | -1.0 to 1.0 |
| 3 | tonality | 3-class | atonal, major, minor |
| 4 | harmonic_style | 7-class | atonal, chromatic, cluster, diatonic, modal, pentatonic, whole_tone |
| 5 | dynamic_shape_m4 | 8-class | crescendo, diminuendo, flat, subito_forte, subito_piano, sustained, swell, terraced |
| 6 | rhythm_style | 6-class | drive, off, ostinato, pulse, rubato, sparse |
| 7 | texture | 5-class | ambient, chamber, full, hybrid, solo |
| 8 | orchestration | 14-label | ambient_pad, brass, choir, electronic, ethnic, guitar, harp, organ, percussion, piano, solo_voice, strings, synth, woodwinds |
"""
        readme_path = os.path.join(cfg["output_dir"], "m3_model_card.md")
        with open(readme_path,"w") as f:
            f.write(card)
        api.upload_file(
            path_or_fileobj=readme_path, path_in_repo="README.md",
            repo_id=HF_M3_REPO, repo_type="model", token=HF_WRITE_TOKEN,
            commit_message="Add M3 model card",
        )
        print("  ✓ Uploaded README.md")
        print(f"  → https://huggingface.co/{HF_M3_REPO}")
    except Exception as e:
        print(f"  ⚠  HF push failed: {e}")


def load_m3_from_hub(cfg: dict) -> str:
    print(f"\n── Downloading M3 from HuggingFace ({HF_M3_REPO}) ──────────────")
    login(token=HF_READ_TOKEN, add_to_git_credential=False)
    local_path = hf_hub_download(
        repo_id=HF_M3_REPO, filename="m3_best.pt",
        token=HF_READ_TOKEN, local_dir=cfg["output_dir"])
    print(f"  ✓ Downloaded → {local_path}")
    return local_path

# =============================================================================
# TRAINING LOOP
# =============================================================================
def run_training(cfg: dict, narrative_vecs: dict) -> Tuple[str, Dataset, list, list, list]:
    torch.manual_seed(cfg["seed"])

    full_ds = MusicDataset(cfg["data_dir"], narrative_vecs)
    if len(full_ds) == 0:
        raise ValueError("Dataset is empty — check data_dir and .pt files")

    # ── Film-level split: ~80% / 10% / 10% of films ──────────────────────
    from collections import defaultdict, Counter

    film_to_indices: Dict[int, List[int]] = defaultdict(list)
    for i in range(len(full_ds)):
        fid = full_ds.samples[i]["film_id"]
        film_to_indices[fid].append(i)

    all_films = sorted(film_to_indices.keys())
    rng       = random.Random(cfg["seed"])
    rng.shuffle(all_films)

    n_films = len(all_films)
    n_test  = max(1, round(0.10 * n_films))
    n_val   = max(1, round(0.10 * n_films))
    n_train = n_films - n_val - n_test

    train_films = all_films[:n_train]
    val_films   = all_films[n_train : n_train + n_val]
    test_films  = all_films[n_train + n_val :]

    train_idx = [i for f in train_films for i in film_to_indices[f]]
    val_idx   = [i for f in val_films   for i in film_to_indices[f]]
    test_idx  = [i for f in test_films  for i in film_to_indices[f]]

    train_ds = torch.utils.data.Subset(full_ds, train_idx)
    val_ds   = torch.utils.data.Subset(full_ds, val_idx)
    test_ds  = torch.utils.data.Subset(full_ds, test_idx)

    print(f"  Film split  : {len(train_films)} train | "
          f"{len(val_films)} val | {len(test_films)} test  "
          f"(total {n_films} films)")
    print(f"  Scene split : {len(train_idx)} train | "
          f"{len(val_idx)} val | {len(test_idx)} test")
    print(f"  Train films : {sorted(train_films)}")
    print(f"  Val   films : {sorted(val_films)}")
    print(f"  Test  films : {sorted(test_films)}")

    assert not (set(train_films) & set(test_films)), "Train/Test film overlap!"
    assert not (set(val_films)   & set(test_films)), "Val/Test film overlap!"
    assert not (set(train_films) & set(val_films)),  "Train/Val film overlap!"
    print("  ✅ Film-level isolation verified — zero overlap across splits")

    # Tonality distribution per split (informational)
    ton_labels = M3_LABEL_SCHEMAS["tonality"]
    for split_name, split_idx in [("train",train_idx),
                                   ("val",  val_idx),
                                   ("test", test_idx)]:
        counts = Counter(full_ds[i]["cls_labels"]["tonality"].item()
                         for i in split_idx)
        dist   = {ton_labels[k] if k >= 0 else "masked": v
                  for k,v in sorted(counts.items())}
        print(f"  {split_name:>5} tonality dist: {dist}")

    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"],
                              shuffle=True,  collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                              shuffle=False, collate_fn=collate_fn, num_workers=0)

    model = MusicDescriptorModule(
        input_dim    = cfg["input_dim"],
        proj_dim     = cfg["proj_dim"],
        head_hidden  = cfg["head_hidden"],
        proj_dropout = cfg["proj_dropout"],
        head_dropout = cfg["head_dropout"],
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable params: {n_params:,}")

    optimizer = AdamW(model.parameters(),
                      lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg["epochs"])

    os.makedirs(cfg["output_dir"], exist_ok=True)
    best_val         = float("inf")
    patience_counter = 0
    ckpt_path        = os.path.join(cfg["output_dir"], cfg["ckpt_name"])

    for epoch in range(1, cfg["epochs"] + 1):

        model.train()
        train_loss = 0.0
        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()
            outputs     = model(batch["combined"].to(device))
            loss, _     = compute_loss(outputs, batch, cfg["focal_gamma"])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            if (step + 1) % 50 == 0:
                print(f"  E{epoch} step {step+1:>3}/{len(train_loader)}"
                      f"  batch_loss={loss.item():.4f}")

        scheduler.step()

        model.eval()
        val_loss         = 0.0
        all_out, all_bat = [], []

        with torch.no_grad():
            for batch in val_loader:
                outputs  = model(batch["combined"].to(device))
                loss, _  = compute_loss(outputs, batch, cfg["focal_gamma"])
                val_loss += loss.item()
                all_out.append({k: v.cpu() for k,v in outputs.items()
                                if k != "music_vector"})
                all_bat.append({
                    "reg_labels": {k: v.cpu() for k,v in batch["reg_labels"].items()},
                    "cls_labels": {k: v.cpu() for k,v in batch["cls_labels"].items()},
                    "ml_labels":  {"orchestration":
                                   batch["ml_labels"]["orchestration"].cpu()},
                })

        avg_train = train_loss / len(train_loader)
        avg_val   = val_loss   / len(val_loader)
        rows      = compute_metrics(all_out, all_bat)

        print_metrics_table(epoch, cfg["epochs"], avg_train, avg_val,
                            rows, best_val, patience_counter,
                            cfg["early_stop_patience"])
        print_scene_preview(model, val_ds, epoch)

        if avg_val < best_val:
            best_val         = avg_val
            patience_counter = 0
            torch.save({
                "epoch":        epoch,
                "model_state":  model.state_dict(),
                "val_loss":     avg_val,
                "cfg":          cfg,
                "train_films":  train_films,
                "val_films":    val_films,
                "test_films":   test_films,
            }, ckpt_path)
            print(f"  ★ Saved best checkpoint → {ckpt_path}  (val={avg_val:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= cfg["early_stop_patience"]:
                print(f"\n  ⏹  Early stopping at epoch {epoch}")
                break

    print(f"\n  Training complete. Best val loss: {best_val:.4f}")
    if cfg.get("push_to_hub", True):
        push_m3_to_hub(cfg, ckpt_path)

    return ckpt_path, full_ds, train_idx, val_idx, test_idx

# =============================================================================
# SAVE MUSIC VECTORS
# NOTE: at inference time, uses predicted labels from .pt files
# not ground truth labels — this is the train/inference gap discussed earlier
# =============================================================================
def save_music_vectors(cfg: dict, ckpt_path: str,
                       scene_vecs: dict, narrative_vecs: dict) -> str:
    print("\n── Generating music_vectors.pt ─────────────────────────────────")

    # For inference we use predicted labels from scene_vectors.pt
    # and narrative_vectors (merged) rather than JSON ground truth
    # because at deployment time there are no GT labels available

    model = MusicDescriptorModule(
        input_dim    = cfg["input_dim"],
        proj_dim     = cfg["proj_dim"],
        head_hidden  = cfg["head_hidden"],
        proj_dropout = cfg["proj_dropout"],
        head_dropout = cfg["head_dropout"],
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    lo_bpm, hi_bpm = M3_LABEL_SCHEMAS["tempo_bpm"]
    lo_val, hi_val = M3_LABEL_SCHEMAS["musical_valence"]
    lo_t,   hi_t   = 1.0, 10.0
    lo_a,   hi_a   = 1.0, 10.0

    N = narrative_vecs["context_vector"].shape[0]

    A: Dict[str,list] = {k:[] for k in [
        "vec","tempo","valence","tonality","harmonic",
        "dynamic","rhythm","texture","orch_probs","orch_bin",
        "scene_id","film_id","position"]}

    M1_CLS_DIMS_ordered = {
        "emotional_valence":4,"conflict_nature":6,"acoustic_space":6,
        "reality_layer":5,"score_dynamic_shape":4,"scene_interaction_tone":5,
    }

    BS = 64
    with torch.no_grad():
        for start in range(0, N, BS):
            end = min(start + BS, N)
            batch_inputs = []

            for i in range(start, end):
                parts = []

                # M1 predicted labels — one-hot cls fields
                for field, dim in M1_CLS_DIMS_ordered.items():
                    oh  = torch.zeros(dim)
                    val = scene_vecs[field][i].item()
                    if 0 <= val < dim: oh[val] = 1.0
                    parts.append(oh)

                # M1 reg scalars
                parts += [
                    scene_vecs["pacing_norm"][i].unsqueeze(0),
                    scene_vecs["action_norm"][i].unsqueeze(0),
                    scene_vecs["tension_norm"][i].unsqueeze(0),
                    scene_vecs["arousal_score"][i].unsqueeze(0),
                ]

                # M1 emotion tags probs
                parts.append(scene_vecs["emotion_tags_probs"][i])

                # M2 predicted labels
                tl = narrative_vecs["tension_level"][i].item()
                al = narrative_vecs["arousal_level"][i].item()
                parts.append(torch.tensor([_norm(tl,lo_t,hi_t)],dtype=torch.float32))
                parts.append(torch.tensor([_norm(al,lo_a,hi_a)],dtype=torch.float32))
                parts.append(narrative_vecs["emotional_shift_trigger"][i].float().unsqueeze(0))

                arc_oh = torch.zeros(5)
                av     = narrative_vecs["narrative_arc_position"][i].item()
                if 0 <= av < 5: arc_oh[av] = 1.0
                parts.append(arc_oh)

                fsh_oh = torch.zeros(4)
                fv     = narrative_vecs["foreshadowing_type"][i].item()
                if 0 <= fv < 4: fsh_oh[fv] = 1.0
                parts.append(fsh_oh)

                tra_oh = torch.zeros(5)
                tv     = narrative_vecs["transition_type"][i].item()
                if 0 <= tv < 5: tra_oh[tv] = 1.0
                parts.append(tra_oh)

                # context_vector
                parts.append(narrative_vecs["context_vector"][i])

                batch_inputs.append(torch.cat(parts, dim=0))

            x   = torch.stack(batch_inputs).to(device)
            out = model(x)

            A["vec"].append(out["music_vector"].cpu())
            A["tempo"].append(  out["pred_tempo"].cpu()   * (hi_bpm-lo_bpm) + lo_bpm)
            A["valence"].append(out["pred_valence"].cpu() * (hi_val-lo_val)/2.0
                                + (hi_val+lo_val)/2.0)
            A["tonality"].append( out["pred_tonality"].cpu().argmax(-1))
            A["harmonic"].append( out["pred_harmonic"].cpu().argmax(-1))
            A["dynamic"].append(  out["pred_dynamic"].cpu().argmax(-1))
            A["rhythm"].append(   out["pred_rhythm"].cpu().argmax(-1))
            A["texture"].append(  out["pred_texture"].cpu().argmax(-1))
            op = torch.sigmoid(out["pred_orch_logits"].cpu())
            A["orch_probs"].append(op)
            A["orch_bin"].append((op > 0.5).long())
            A["scene_id"].append(narrative_vecs["scene_id"][start:end])
            A["film_id"].append( narrative_vecs["film_id"][start:end])
            A["position"].append(narrative_vecs["position_in_film"][start:end])

    C = {k: torch.cat(v) for k,v in A.items()}

    out_path = os.path.join(cfg["output_dir"], "music_vectors.pt")
    torch.save({
        "music_vector":        C["vec"],
        "tempo_bpm":           C["tempo"],
        "musical_valence":     C["valence"],
        "tonality":            C["tonality"],
        "harmonic_style":      C["harmonic"],
        "dynamic_shape_m4":    C["dynamic"],
        "rhythm_style":        C["rhythm"],
        "texture":             C["texture"],
        "orchestration_probs": C["orch_probs"],
        "orchestration":       C["orch_bin"],
        "scene_id":            C["scene_id"],
        "film_id":             C["film_id"],
        "position_in_film":    C["position"],
    }, out_path)
    print(f"  ✓ Saved {C['vec'].shape[0]} music vectors → {out_path}")
    return out_path

# =============================================================================
# SANITY CHECK
# =============================================================================
def sanity_check(out_path: str):
    data = torch.load(out_path)
    N    = data["music_vector"].shape[0]
    W    = 66
    print(f"\n── Sanity Check — music_vectors.pt {'─'*(W-35)}")
    print(f"  Total scenes  : {N}")
    print(f"  {'Key':<26} {'Shape':<18} Dtype")
    print(f"  {'─'*W}")
    for k,v in data.items():
        print(f"  {k:<26} {str(tuple(v.shape)):<18} {v.dtype}")
    print(f"  {'─'*W}")
    print(f"  {len(data)} fields ✓\n")

# =============================================================================
# ENTRY POINT
# =============================================================================
def run(cfg: dict = CFG):
    print("=" * 66)
    print("  MODULE 3 — Music Descriptor Prediction")
    print("  Input: 313-d (M1 labels + M2 labels + context_vector)")
    print("  8 heads | MLP | focal loss on orchestration")
    print("=" * 66)

    # Step 1 — ensure scene_vectors.pt exists (download M1 + run if missing)
    sv_path    = _ensure_scene_vectors(cfg)
    scene_vecs = torch.load(sv_path, map_location="cpu")
    print(f"  ✓ scene_vectors: {scene_vecs['scene_vector'].shape[0]} scenes")

    # Step 2 — ensure narrative_vectors exist (download M2 + run if missing)
    print(f"\n  Checking narrative_vectors ...")
    narrative_vecs = _ensure_narrative_vectors(cfg, scene_vecs)

    # Step 3 — train M3 or load from Hub
    if cfg.get("load_from_hub", False):
        ckpt_path = load_m3_from_hub(cfg)

        from collections import defaultdict
        full_ds = MusicDataset(cfg["data_dir"], narrative_vecs)

        film_to_indices: Dict[int, List[int]] = defaultdict(list)
        for i in range(len(full_ds)):
            fid = full_ds.samples[i]["film_id"]
            film_to_indices[fid].append(i)

        all_films = sorted(film_to_indices.keys())
        rng       = random.Random(cfg["seed"])
        rng.shuffle(all_films)

        n_films = len(all_films)
        n_test  = max(1, round(0.10 * n_films))
        n_val   = max(1, round(0.10 * n_films))
        n_train = n_films - n_val - n_test

        train_films = all_films[:n_train]
        val_films   = all_films[n_train : n_train + n_val]
        test_films  = all_films[n_train + n_val :]

        train_idx = [i for f in train_films for i in film_to_indices[f]]
        val_idx   = [i for f in val_films   for i in film_to_indices[f]]
        test_idx  = [i for f in test_films  for i in film_to_indices[f]]

        print(f"  Film split (load_from_hub): "
              f"{len(train_films)} train | {len(val_films)} val | "
              f"{len(test_films)} test films")

        saved_ckpt = torch.load(ckpt_path, map_location="cpu")
        if "test_films" in saved_ckpt:
            assert sorted(saved_ckpt["test_films"]) == sorted(test_films), \
                "Test film mismatch between checkpoint and rebuilt split!"
            print("  ✅ Film split verified against checkpoint")
        else:
            print("  ⚠  Checkpoint has no saved film lists — verify manually")
    else:
        ckpt_path, full_ds, train_idx, val_idx, test_idx = run_training(
            cfg, narrative_vecs)

    # Step 4 — full inference → music_vectors.pt
    out_path = save_music_vectors(cfg, ckpt_path, scene_vecs, narrative_vecs)
    sanity_check(out_path)
    print(f"  ✓ Done. music_vectors.pt ready for downstream use.")


run(CFG)

Test

In [ ]:
# =============================================================================
# MODULE 3 — TEST SET EVALUATION
# =============================================================================

import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    mean_absolute_error, r2_score,
    precision_recall_fscore_support,
)
from collections import Counter

# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt_path = "/kaggle/working/m3_best.pt"
ckpt      = torch.load(ckpt_path, map_location=device)

model_test = MusicDescriptorModule(
    input_dim    = CFG["input_dim"],
    proj_dim     = CFG["proj_dim"],
    head_hidden  = CFG["head_hidden"],
    proj_dropout = CFG["proj_dropout"],
    head_dropout = CFG["head_dropout"],
).to(device)

model_test.load_state_dict(ckpt["model_state"])
model_test.eval()
print(f"  Loaded checkpoint from epoch {ckpt['epoch']}")
print(f"  Val loss : {ckpt['val_loss']:.4f}\n")

# ── Rebuild exact same split ──────────────────────────────────────────────────
# Load merged narrative vectors
parts = []
for s in ["train", "val", "test"]:
    parts.append(torch.load(
        f"/kaggle/working/narrative_vectors_{s}.pt", map_location="cpu"))
tensor_keys    = [k for k in parts[0] if isinstance(parts[0][k], torch.Tensor)]
narrative_vecs = {k: torch.cat([p[k] for p in parts], dim=0) for k in tensor_keys}

from collections import defaultdict

full_ds = MusicDataset(CFG["data_dir"], narrative_vecs)

# ── Rebuild exact film-level split (must match run_training exactly) ──────
film_to_indices: Dict[int, List[int]] = defaultdict(list)
for i in range(len(full_ds)):
    fid = full_ds.samples[i]["film_id"]
    film_to_indices[fid].append(i)

all_films = sorted(film_to_indices.keys())
rng       = random.Random(CFG["seed"])
rng.shuffle(all_films)

n_films = len(all_films)
n_test  = max(1, round(0.10 * n_films))
n_val   = max(1, round(0.10 * n_films))
n_train = n_films - n_val - n_test

train_films = all_films[:n_train]
val_films   = all_films[n_train : n_train + n_val]
test_films  = all_films[n_train + n_val :]

train_idx = [i for f in train_films for i in film_to_indices[f]]
val_idx   = [i for f in val_films   for i in film_to_indices[f]]
test_idx  = [i for f in test_films  for i in film_to_indices[f]]

print(f"  Train films ({len(train_films)}): {sorted(train_films)}")
print(f"  Val   films ({len(val_films)}):   {sorted(val_films)}")
print(f"  Test  films ({len(test_films)}):  {sorted(test_films)}")

# ── Verify against checkpoint ─────────────────────────────────────────────
if "test_films" in ckpt:
    assert sorted(ckpt["test_films"]) == sorted(test_films), \
        f"Test film mismatch!\n  ckpt: {sorted(ckpt['test_films'])}\n  rebuilt: {sorted(test_films)}"
    print("  ✅ Film split verified against checkpoint")
else:
    print("  ⚠  Checkpoint has no saved film lists — verify split manually")

test_ds     = torch.utils.data.Subset(full_ds, test_idx)
test_loader = DataLoader(
    test_ds, batch_size=CFG["batch_size"],
    shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f"  Test set size : {len(test_ds)} samples\n")

# ── Run inference ─────────────────────────────────────────────────────────────
tmp_p, tmp_t = [], []
val_p, val_t = [], []
cls_p = {k: [] for k in M3_CLS_HEADS}
cls_t = {k: [] for k in M3_CLS_HEADS}
orc_p, orc_t = [], []

lo_bpm, hi_bpm = M3_LABEL_SCHEMAS["tempo_bpm"]
lo_val, hi_val = M3_LABEL_SCHEMAS["musical_valence"]

with torch.no_grad():
    for batch in test_loader:
        out = model_test(batch["combined"].to(device))

        # Regression
        tp = out["pred_tempo"].cpu().numpy() * (hi_bpm - lo_bpm) + lo_bpm
        tt = batch["reg_labels"]["tempo_bpm"].numpy() * (hi_bpm - lo_bpm) + lo_bpm
        tmp_p.extend(tp.tolist()); tmp_t.extend(tt.tolist())

        vp = out["pred_valence"].cpu().numpy() * (hi_val - lo_val) / 2.0 \
             + (hi_val + lo_val) / 2.0
        vt = batch["reg_labels"]["musical_valence"].numpy() \
             * (hi_val - lo_val) + lo_val
        val_p.extend(vp.tolist()); val_t.extend(vt.tolist())

        # Classification
        for field, pred_key in [
            ("tonality",         "pred_tonality"),
            ("harmonic_style",   "pred_harmonic"),
            ("dynamic_shape_m4", "pred_dynamic"),
            ("rhythm_style",     "pred_rhythm"),
            ("texture",          "pred_texture"),
        ]:
            tgt  = batch["cls_labels"][field].numpy()
            mask = tgt >= 0
            if mask.sum() == 0:
                continue
            preds = out[pred_key].cpu()[mask].argmax(-1).numpy()
            cls_p[field].extend(preds.tolist())
            cls_t[field].extend(tgt[mask].tolist())

        # Multi-label
        pp = (torch.sigmoid(out["pred_orch_logits"]).cpu() > 0.5).int().numpy()
        tt = batch["ml_labels"]["orchestration"].int().numpy()
        orc_p.append(pp); orc_t.append(tt)

# =============================================================================
# PRINT RESULTS
# =============================================================================
W  = 112
EQ = "═" * W
DH = "─" * W

print(f"\n{EQ}")
print(f"  MODULE 3 — TEST SET RESULTS")
print(f"{EQ}")

# ── Regression ────────────────────────────────────────────────────────────────
print(f"\n  📈 REGRESSION HEADS")
print(f"  {DH}")
print(f"  {'Head':<24} {'MAE':>10} {'R²':>10} {'Min Pred':>12} {'Max Pred':>12}")
print(f"  {'─'*72}")

for label, pp, tt in [
    ("tempo_bpm",       tmp_p, tmp_t),
    ("musical_valence", val_p, val_t),
]:
    mae = mean_absolute_error(tt, pp)
    r2  = r2_score(tt, pp)
    print(f"  {label:<24} {mae:>10.4f} {r2:>10.4f} "
          f"{min(pp):>12.4f} {max(pp):>12.4f}")

# ── Classification heads ──────────────────────────────────────────────────────
for field in M3_CLS_HEADS:
    if not cls_t[field]:
        continue

    labels      = M3_LABEL_SCHEMAS[field]
    present_idx = sorted(set(cls_t[field]))
    present_lbl = [labels[i] for i in present_idx]

    acc      = accuracy_score(cls_t[field], cls_p[field])
    f1_macro = f1_score(cls_t[field], cls_p[field],
                        average="macro",    zero_division=0)
    f1_wtd   = f1_score(cls_t[field], cls_p[field],
                        average="weighted", zero_division=0)

    print(f"\n  📊 CLASSIFICATION HEAD — {field}")
    print(f"  {DH}")
    print(f"  Accuracy   : {acc:.4f}")
    print(f"  F1 Macro   : {f1_macro:.4f}")
    print(f"  F1 Weighted: {f1_wtd:.4f}")
    print(f"  Support    : {len(cls_t[field])} samples")

    f1_per  = f1_score(cls_t[field], cls_p[field],
                       average=None, labels=present_idx, zero_division=0)
    pr_per, rc_per, _, sup_per = precision_recall_fscore_support(
        cls_t[field], cls_p[field],
        average=None, labels=present_idx, zero_division=0)

    print(f"\n  {'Class':<24} {'Precision':>10} {'Recall':>10} "
          f"{'F1':>10} {'Support':>10}")
    print(f"  {'─'*68}")

    cnt = Counter(cls_t[field])
    for i, (lbl, f1_val, pr_val, rc_val) in enumerate(
            zip(present_lbl, f1_per, pr_per, rc_per)):
        support = cnt[present_idx[i]]
        bar     = "█" * int(f1_val * 20)
        print(f"  {lbl:<24} {pr_val:>10.4f} {rc_val:>10.4f} "
              f"{f1_val:>10.4f} {support:>10}   {bar}")

    print(f"\n  Detailed report:")
    print(classification_report(
        cls_t[field], cls_p[field],
        labels=present_idx,
        target_names=present_lbl,
        zero_division=0,
        digits=4,
    ))

# ── Multi-label — orchestration ───────────────────────────────────────────────
print(f"\n  🎼 MULTI-LABEL HEAD — orchestration")
print(f"  {DH}")

p = np.vstack(orc_p)
t = np.vstack(orc_t)

f1_macro = f1_score(t, p, average="macro",    zero_division=0)
f1_wtd   = f1_score(t, p, average="weighted", zero_division=0)
f1_per   = f1_score(t, p, average=None,        zero_division=0)

print(f"  F1 Macro   : {f1_macro:.4f}")
print(f"  F1 Weighted: {f1_wtd:.4f}\n")
print(f"  {'Instrument':<20} {'F1':>10} {'Support':>10}")
print(f"  {'─'*44}")

support_per = t.sum(axis=0).astype(int)
orch_labels = M3_LABEL_SCHEMAS["orchestration"]
for lbl, f1_val, sup in zip(orch_labels, f1_per, support_per):
    bar = "█" * int(f1_val * 20)
    print(f"  {lbl:<20} {f1_val:>10.4f} {sup:>10}   {bar}")

print(f"\n{EQ}")
print(f"  ✓ Module 3 test evaluation complete")
print(f"{EQ}\n")